# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbdulWasay65/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

### Method choice

I will use **Logistic Regression** for this modeling lane.

The target is binary: whether a page achieves at least a 2-position improvement from March to April 2026. Logistic Regression fits this target because it estimates the probability of the positive outcome and provides an interpretable baseline model.

I will use March 2026 decision-time signals as model inputs and keep the April outcome only as the evaluation target. The main features will come from signals available at decision time, such as search impressions, clicks, average position, and available traffic/engagement signals where appropriate.

Logistic Regression is preferred over a more complex model at this stage because the goal is to establish an interpretable model that can be compared fairly against the Week-4 rule-based baseline. A more complex model would only be useful if it provides measurable improvement on the same validation design and metric.

The model output will be treated as **directional decision support**, not as a causal estimate or guarantee that a page will improve after a refresh.


In [7]:
# This cell is for CODE
# Section 1 — Inspect the modeling data prepared above

print("Available Python variables:")
for name in sorted([k for k in globals().keys() if not k.startswith("_")]):
    if name in [
        "con", "rel",
        "march_pages", "matched_pages",
        "model_df", "features_df", "target_df",
        "baseline_df"
    ]:
        print(name)

print("\nDuckDB connection:", "available" if "con" in globals() else "missing")
print("Warehouse relation:", rel if "rel" in globals() else "missing")
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Available Python variables:
con
rel

DuckDB connection: available
Warehouse relation: hf://datasets/FlyRank/internship-warehouse


In [8]:
# Section 1B — Inspect available warehouse data

import duckdb
from google.colab import userdata

# Create/recreate DuckDB connection
con = duckdb.connect()

# Load Hugging Face token
HF_TOKEN = userdata.get("HF_TOKEN")

# Configure Hugging Face access
con.execute(
    f"""
    CREATE OR REPLACE SECRET hf_secret
    (TYPE huggingface, TOKEN '{HF_TOKEN}')
    """
)

# Warehouse relation
rel = "hf://datasets/FlyRank/internship-warehouse"

# Main performance data path
performance_path = (
    f"{rel}/fact_content_daily_performance/**/*.parquet"
)

print("Warehouse connection configured.")
print("Warehouse relation:", rel)

# Check that the performance data can be read
check_query = f"""
SELECT
    COUNT(*) AS rows_available
FROM read_parquet('{performance_path}')
WHERE month = '2026-03'
"""

check_result = con.sql(check_query).df()

print("March 2026 rows available:", int(check_result["rows_available"].iloc[0]))

Warehouse connection configured.
Warehouse relation: hf://datasets/FlyRank/internship-warehouse
March 2026 rows available: 9841378


In [9]:
# Section 1C — Inspect March-to-April outcome availability

outcome_query = f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,
        AVG(gsc_avg_position) AS march_avg_position
    FROM read_parquet(
        '{performance_path}'
    )
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions,
        SUM(gsc_clicks) AS april_clicks,
        AVG(gsc_avg_position) AS april_avg_position
    FROM read_parquet(
        '{performance_path}'
    )
    WHERE month = '2026-04'
      AND gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    COUNT(*) AS march_pages,
    COUNT(april.content_hash_id) AS pages_with_april_data,
    COUNT(*) - COUNT(april.content_hash_id) AS pages_without_april_data
FROM march
LEFT JOIN april
    ON march.client_hash_id = april.client_hash_id
   AND march.content_hash_id = april.content_hash_id
"""

outcome_check = con.sql(outcome_query).df()

display(outcome_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_pages,pages_with_april_data,pages_without_april_data
0,176738,158549,18189


## 2. Split design

### Split design

I will use a **grouped train/test split by client** so that pages from the same client do not appear in both training and test sets.

This is an honest validation design for the content-refresh prioritization lane because pages from the same client can share site-level characteristics. Keeping each client entirely within one split reduces the risk that the model benefits from seeing closely related client data during training.

The model will use only March 2026 decision-time features. April 2026 performance is used only to define the evaluation target and is not provided to the model as an input.

I will use an 80/20 grouped split with a fixed random seed so the result is reproducible.



In [10]:
# Section 2A — Prepare the modeling dataframe

import pandas as pd
import numpy as np

# March 2026 = decision-time features
# April 2026 = outcome used only to create the evaluation target

model_query = f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        AVG(gsc_avg_position) AS avg_position
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        AVG(gsc_avg_position) AS april_avg_position
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-04'
      AND gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.impressions,
    m.clicks,
    m.avg_position,
    a.april_avg_position,

    CASE
        WHEN a.april_avg_position <= m.avg_position - 2
        THEN 1
        ELSE 0
    END AS target

FROM march m
INNER JOIN april a
    ON m.client_hash_id = a.client_hash_id
   AND m.content_hash_id = a.content_hash_id

WHERE m.avg_position IS NOT NULL
  AND a.april_avg_position IS NOT NULL
"""

model_df = con.sql(model_query).df()

print("Modeling rows available:", len(model_df))
print("Positive target rate:", round(model_df["target"].mean(), 4))

display(model_df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modeling rows available: 158549
Positive target rate: 0.2283


,client_hash_id,content_hash_id,impressions,clicks,avg_position,april_avg_position,target
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,7.209549,6.098680,0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,2.987198,5.378531,0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.724039,6.533080,0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.244844,6.423190,0
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,4.209227,5.480430,0


In [11]:
# Section 2B — Grouped train/test split

from sklearn.model_selection import GroupShuffleSplit

feature_cols = [
    "impressions",
    "clicks",
    "avg_position"
]

X = model_df[feature_cols].copy()
y = model_df["target"].copy()
groups = model_df["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_groups = groups.iloc[train_idx]
test_groups = groups.iloc[test_idx]

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("Training clients:", train_groups.nunique())
print("Test clients:", test_groups.nunique())

print(
    "Clients overlapping between train/test:",
    len(set(train_groups) & set(test_groups))
)

print("Training positive rate:", round(y_train.mean(), 4))
print("Test positive rate:", round(y_test.mean(), 4))

Training rows: 137445
Test rows: 21104
Training clients: 36
Test clients: 10
Clients overlapping between train/test: 0
Training positive rate: 0.2454
Test positive rate: 0.117


In [12]:
# Section 2C — Create the client-grouped train/test split

from sklearn.model_selection import GroupShuffleSplit

# Use the modeling dataframe created earlier
print("Modeling rows available:", len(model_df))

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        model_df,
        model_df["target"],
        groups=model_df["client_hash_id"]
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

print("Train clients:", train_df["client_hash_id"].nunique())
print("Test clients:", test_df["client_hash_id"].nunique())

client_overlap = set(train_df["client_hash_id"]) & set(test_df["client_hash_id"])

print("Client overlap:", len(client_overlap))

print(
    "Grouped split check:",
    "PASS" if len(client_overlap) == 0 else "FAIL"
)

Modeling rows available: 158549
Train rows: 137445
Test rows: 21104
Train clients: 36
Test clients: 10
Client overlap: 0
Grouped split check: PASS


## 3. Train + compare vs my baseline

### Modeling approach

I will train a Logistic Regression model using March 2026 decision-time signals.

The model will estimate the probability that a page achieves at least a 2-position improvement in April. The model will be evaluated on the held-out client-grouped test set.

The Week-4 baseline and the Logistic Regression model will be evaluated on the **same test pages** and against the **same binary target** using Average Precision as the primary ranking metric.

Average Precision is used because the positive class represents only about 22.83% of the matched pages. It therefore provides a more informative measure of how well the ranking concentrates positive cases near the top than simple accuracy.

The model is intended as a decision-support ranking model rather than a causal prediction of whether a content refresh will produce an improvement.


In [13]:
# This cell is for CODE
# Section 3 — Train model and compare with Week-4 baseline

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score
import pandas as pd
import numpy as np

# Train a deliberately constrained Random Forest
model = RandomForestClassifier(
    n_estimators=150,
    max_depth=6,
    min_samples_leaf=50,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

model.fit(X_train, y_train)

# Model probability scores
model_scores = model.predict_proba(X_test)[:, 1]

# ---------------------------------------------------------
# Week-4 baseline score on the SAME test rows
# ---------------------------------------------------------

baseline_test = model_df.iloc[test_idx].copy()

baseline_scores = (
    np.log1p(baseline_test["impressions"].clip(lower=0))
    + baseline_test["avg_position"].clip(upper=20)
)

# ---------------------------------------------------------
# Same metric: Average Precision
# ---------------------------------------------------------

model_ap = average_precision_score(y_test, model_scores)
baseline_ap = average_precision_score(y_test, baseline_scores)

comparison = pd.DataFrame({
    "method": [
        "Week-4 rule baseline",
        "Random Forest"
    ],
    "average_precision": [
        baseline_ap,
        model_ap
    ]
})

comparison["improvement_vs_baseline"] = (
    comparison["average_precision"] - baseline_ap
)

display(comparison)

print("Baseline AP:", round(baseline_ap, 4))
print("Random Forest AP:", round(model_ap, 4))
print(
    "Model improvement:",
    round(model_ap - baseline_ap, 4)
)

,method,average_precision,improvement_vs_baseline
0,Week-4 rule baseline,0.208649,0.000000
1,Random Forest,0.368537,0.159888


Baseline AP: 0.2086
Random Forest AP: 0.3685
Model improvement: 0.1599


## 4. Errors and interpretation

### Error analysis

The Random Forest achieved higher Average Precision than the Week-4 rule baseline on the same grouped test split. The model's feature importance is dominated by `avg_position` (80.8%), followed by `impressions` (12.5%) and `clicks` (6.8%).

The main observed error pattern is a high number of false positives. The model predicted 8,506 positive cases while 2,469 were actually positive, producing 6,469 false positives and 432 false negatives. Many false positives have very weak average positions, suggesting that the model treats poor ranking as a strong opportunity signal even when the target outcome does not occur.

The relatively small number of false negatives suggests that the model captures most observed positives, but some positive cases near the decision boundary are missed. Several false negatives have substantial impressions or clicks, showing that visibility and traffic signals alone do not guarantee correct classification.

Overall, the model appears to trade precision for broader positive-case coverage. Its higher Average Precision is encouraging, but the false-positive volume means the model should be treated as decision support rather than an automatic refresh decision.


In [14]:
# This cell is for CODE
# Section 4 — Errors and interpretation

import pandas as pd
import numpy as np

# Recreate the modeling feature list used by the Random Forest
feature_cols = [
    "impressions",
    "clicks",
    "avg_position"
]

# Confirm the trained model and test data exist
required_objects = ["model", "X_test", "y_test"]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Missing required Section 3 objects: "
        + ", ".join(missing_objects)
        + ". Run Section 3 before Section 4."
    )

# Feature importance from the trained Random Forest
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("Feature importance:")
display(importance_df)

# Predictions on the test set
test_predictions = model.predict(X_test)
test_scores = model.predict_proba(X_test)[:, 1]

# Build error-analysis dataframe
error_df = X_test.copy()

# Reset indexes so values align correctly
error_df = error_df.reset_index(drop=True)

y_test_reset = pd.Series(y_test).reset_index(drop=True)

error_df["actual"] = y_test_reset
error_df["model_score"] = test_scores
error_df["predicted"] = test_predictions

# False positives
false_positives = error_df[
    (error_df["actual"] == 0) &
    (error_df["predicted"] == 1)
].sort_values(
    "model_score",
    ascending=False
)

print("\nFalse positives:", len(false_positives))

display(
    false_positives[
        [
            "impressions",
            "clicks",
            "avg_position",
            "actual",
            "model_score",
            "predicted"
        ]
    ].head(10)
)

# False negatives
false_negatives = error_df[
    (error_df["actual"] == 1) &
    (error_df["predicted"] == 0)
].sort_values(
    "model_score",
    ascending=False
)

print("\nFalse negatives:", len(false_negatives))

display(
    false_negatives[
        [
            "impressions",
            "clicks",
            "avg_position",
            "actual",
            "model_score",
            "predicted"
        ]
    ].head(10)
)

# Summary
print("\nError analysis summary:")
print("Test rows:", len(error_df))
print("Actual positives:", int(error_df["actual"].sum()))
print("Predicted positives:", int(error_df["predicted"].sum()))
print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

Feature importance:


,feature,importance
2,avg_position,0.807765
0,impressions,0.124598
1,clicks,0.067637



False positives: 6469


,impressions,clicks,avg_position,actual,model_score,predicted
18696,3.0,0.0,80.333333,0,0.892995,1
9700,11.0,0.0,90.900000,0,0.880929,1
10459,9.0,0.0,78.312500,0,0.877478,1
9442,8.0,0.0,72.900000,0,0.877394,1
10360,12.0,0.0,74.411111,0,0.873391,1
9294,10.0,0.0,69.250000,0,0.871969,1
18452,17.0,0.0,88.525641,0,0.870110,1
19105,20.0,0.0,76.976190,0,0.862901,1
20688,17.0,0.0,68.896667,0,0.861975,1
7119,25.0,0.0,75.619048,0,0.861212,1



False negatives: 432


,impressions,clicks,avg_position,actual,model_score,predicted
16656,1499.0,1.0,18.159575,1,0.499806,0
1536,1219.0,1.0,17.981001,1,0.499563,0
11817,8507.0,24.0,11.630615,1,0.498403,0
9185,863.0,0.0,21.290310,1,0.498266,0
1586,827.0,0.0,17.691424,1,0.498168,0
1318,1441.0,0.0,22.068311,1,0.497879,0
6140,7713.0,4.0,13.462188,1,0.497553,0
11789,16846.0,82.0,10.188309,1,0.497287,0
4349,1358.0,10.0,16.899810,1,0.497056,0
1571,2394.0,5.0,18.620448,1,0.496402,0



Error analysis summary:
Test rows: 21104
Actual positives: 2469
Predicted positives: 8506
False positives: 6469
False negatives: 432


## Self-check


- Every section is filled with both Markdown reasoning and supporting code.
- The notebook runs successfully from top to bottom without errors.
- The method choice is justified for the content-refresh prioritization lane.
- The split is grouped by client, so clients do not overlap between training and test sets.
- The model and Week-4 baseline are evaluated on the same test data using Average Precision.
- The Random Forest achieved an observed Average Precision of 0.3685 compared with 0.2086 for the Week-4 baseline.
- The model showed an observed improvement of 0.1599 Average Precision over the baseline.
- Feature importance was interpreted rather than treating complexity as automatically better.
- False positives and false negatives were reviewed and their limitations were discussed.
- Claims are described as observed, measured, directional, and decision-support rather than causal.
- No client names, URLs, or private queries are included.
- No future-window information is used as a model input.
- The notebook is ready to be executed top to bottom and committed under `work/notebooks/w05_model.ipynb`.